In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import PowerTransformer

In [40]:
df = pd.read_csv('/content/drive/MyDrive/colab/mlpr/dataset.csv')

In [ ]:
#display(df.columns)

Index(['timestamp', 'pm25_ugm3', 'pm10_ugm3', 'no_ugm3', 'no2_ugm3', 'nox_ppb',
       'nh3_ugm3', 'so2_ugm3', 'co_mgm3', 'ozone_ugm3', 'at_c', 'rh_pct',
       'ws_ms', 'wd_deg', 'rf_mm', 'tot_rf_mm', 'sr_wm2', 'state', 'city',
       'station', 'station_id', 'latitude', 'longitude', 'era5_sw_down_wm2',
       'era5_dewpoint_k', 'era5_pressure_pa', 'era5_temp_k',
       'era5_cloud_cover', 'era5_precip_m', 'era5_u10_ms', 'era5_v10_ms',
       'era5_temp_c', 'era5_dewpoint_c', 'era5_pressure_hpa', 'era5_precip_mm',
       'era5_wind_speed_ms', 'era5_wind_dir_deg', 'era5_vpd_kpa',
       'era5_rh_pct', 'solar_altitude_deg', 'solar_zenith_deg', 'cos_zenith'],
      dtype='object')

In [ ]:
#display(df.head())

,pm25_ugm3,pm10_ugm3,no_ugm3,no2_ugm3,nox_ppb,nh3_ugm3,so2_ugm3,co_mgm3,ozone_ugm3,latitude,...,solar_zenith_deg,cos_zenith,station_id_encoded,station_encoded,state_encoded,city_encoded,year,month,day,hour
0,19.084999,38.794998,14.382500,4.947500,14.322500,10.155000,3.480000,0.762500,4.765000,13.204880,...,85.332840,0.081367,360.158081,360.158081,269.334015,360.158081,2023,1,1,7
1,45.410000,64.642502,5.647500,13.365000,18.982500,5.905000,1.222500,0.292500,11.787500,11.875000,...,87.967094,0.035473,259.040100,259.040100,221.699615,259.040100,2023,1,1,7
2,77.750000,164.000000,12.100000,67.025002,45.474998,53.200001,44.349998,0.172500,4.533333,26.088131,...,83.800438,0.107992,140.013062,140.013062,112.064301,140.013062,2023,1,1,7
3,52.500000,79.000000,4.012735,23.902308,12.866176,9.081314,4.175000,0.730566,3.733333,27.308329,...,86.959023,0.053050,26.129847,26.129847,112.064301,26.129847,2023,1,1,7
4,62.250000,186.250000,0.425000,30.600000,16.600000,25.197840,12.875000,0.375000,36.866665,25.376776,...,84.595512,0.094186,104.376892,104.376892,112.064301,104.376892,2023,1,1,7


In [ ]:
#df.drop(columns=['latitude', 'longitude', 'station_id', 'station', 'state', 'city'],  inplace=True)
from sklearn.preprocessing import LabelEncoder


target_encoders = {}

for col in ['station_id', 'station', 'state', 'city']:
    if col in df.columns:
        target_encoders[col] = df.groupby(col)['sr_wm2'].mean().to_dict()

# Step 2: Apply encoding to entire dataset
train_mask = df['year'] < 2025
for col in ['station_id', 'station', 'state', 'city']:
    encoder = df.loc[train_mask].groupby(col)['sr_wm2'].mean().to_dict()
    global_mean = df.loc[train_mask, 'sr_wm2'].mean()
    df[f'{col}_encoded'] = df[col].map(encoder).fillna(global_mean)


float_cols = df.select_dtypes(include=['float64']).columns
df[float_cols] = df[float_cols].astype('float32')

df['timestamp'] = pd.to_datetime(df['timestamp'])

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2549291 entries, 0 to 2549290
Data columns (total 46 columns):
 #   Column              Dtype         
---  ------              -----         
 0   timestamp           datetime64[ns]
 1   pm25_ugm3           float32       
 2   pm10_ugm3           float32       
 3   no_ugm3             float32       
 4   no2_ugm3            float32       
 5   nox_ppb             float32       
 6   nh3_ugm3            float32       
 7   so2_ugm3            float32       
 8   co_mgm3             float32       
 9   ozone_ugm3          float32       
 10  at_c                float32       
 11  rh_pct              float32       
 12  ws_ms               float32       
 13  wd_deg              float32       
 14  rf_mm               float32       
 15  tot_rf_mm           float32       
 16  sr_wm2              float32       
 17  state               object        
 18  city                object        
 19  station             object        
 20  st

In [42]:
# Make sure rows are in chronological order
df = df.sort_values("timestamp").reset_index(drop=True)

# Feature engineering
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day
df["hour"] = df["timestamp"].dt.hour

# Drop obvious redundancies
cols_to_drop = [
    'timestamp',                    # Already extracted as year/month/day/hour
    'at_c', 'rh_pct', 'ws_ms', 'wd_deg', 'rf_mm', 'tot_rf_mm',  # Duplicated in ERA5
    'station_id', 'station', 'state', 'city',  # Keep encoded versions only
    'era5_temp_k', 'era5_dewpoint_k', 'era5_pressure_pa',  # Keep _c and _hpa versions
    'era5_u10_ms', 'era5_v10_ms',   # Use era5_wind_speed_ms instead
    'era5_precip_m',                # Keep era5_precip_mm
    'solar_altitude_deg',           # Keep solar_zenith_deg (more informative for prediction)
    'era5_sw_down_wm2',             # Target leakage!
]

df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
# 1. Cyclical Time Encoding (Fixing the 11 PM to Midnight jump)
df['hour_sin'] = np.sin(df['hour'] * (2. * np.pi / 24)).astype('float32')
df['hour_cos'] = np.cos(df['hour'] * (2. * np.pi / 24)).astype('float32')
df['month_sin'] = np.sin((df['month'] - 1) * (2. * np.pi / 12)).astype('float32')
df['month_cos'] = np.cos((df['month'] - 1) * (2. * np.pi / 12)).astype('float32')

# Drop the raw hour and month columns as they are no longer needed
df = df.drop(columns=['hour', 'month'])

# 2. Sort Chronologically AND Geographically (CRITICAL)
df = df.sort_values(by=['station_id_encoded', 'year', 'hour_sin'])

# Drop the rows at the top of each station's timeline that now have NaNs due to shifting
df = df.dropna()

In [45]:
# 1. Normalize the Target Variable (Yeo-Johnson Power Transform)
pt = PowerTransformer(method='yeo-johnson')
df['sr_wm2_scaled'] = pt.fit_transform(df[['sr_wm2']]).astype('float32')

# 2. Chronological Split (e.g., Train on years < 2023, Test on 2023)
train_df = df[df['year'] < 2025]
test_df = df[df['year'] >= 2025]

# Separate features (X) and target (y)
features = [col for col in df.columns if col not in ['sr_wm2', 'sr_wm2_scaled', 'year']]

X_train = train_df[features]
y_train = train_df['sr_wm2_scaled']

X_test = test_df[features]
y_test = test_df['sr_wm2_scaled']

print(f"Training rows: {len(X_train)} | Testing rows: {len(X_test)}")

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:188: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


Training rows: 1694822 | Testing rows: 849069


In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
SEQUENCE_LENGTH = 24   # look-back window (hours)
BATCH_SIZE      = 256
FEATURES        = [col for col in df.columns
                   if col not in ['sr_wm2', 'sr_wm2_scaled', 'year']]
N_FEATURES      = len(FEATURES)

# ── STEP 1 : CHRONOLOGICAL TRAIN / TEST SPLIT ────────────────────────────────
# Do this BEFORE building sequences so no test rows ever enter a training window
train_df = df[df['year'] < 2025].copy()
test_df  = df[df['year'] >= 2025].copy()

print(f"Train rows : {len(train_df):,}")
print(f"Test rows  : {len(test_df):,}")
print(f"Stations   : {train_df['station_id'].nunique()}")

# ── STEP 2 : FEATURE SCALING (fit on train only) ─────────────────────────────
# Scale features independently of the target transform done earlier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_df[FEATURES] = scaler.fit_transform(train_df[FEATURES]).astype('float32')
test_df[FEATURES]  = scaler.transform(test_df[FEATURES]).astype('float32')

# ── HELPER : build a tf.data.Dataset for one DataFrame ───────────────────────
def build_station_dataset(station_df, sequence_length, batch_size, shuffle=False):
    """
    Build a per-station sliding-window dataset and concatenate them.

    Why per-station?
      timeseries_dataset_from_array treats the input as ONE continuous series.
      If you stack station A then station B, the final SEQUENCE_LENGTH rows of A
      become the look-back context for station B's first prediction — the model
      learns a spurious cross-station dependency that doesn't exist at inference.

    Strategy:
      1. Sort each station chronologically.
      2. Build a tf.data.Dataset for that station alone.
      3. Concatenate all station datasets.
      4. Shuffle + batch at the end (so sequences from different stations mix
         during training, which is desirable for generalisation).
    """
    station_datasets = []
    skipped = 0

    for sid, grp in station_df.groupby('station_id_encoded', sort=False):
        grp = grp.sort_values('timestamp') if 'timestamp' in grp.columns \
              else grp   # already sorted earlier; keep as safety net

        X = grp[FEATURES].values.astype('float32')
        y = grp['sr_wm2_scaled'].values.astype('float32')

        # Need at least (sequence_length + 1) rows to form one sample
        if len(X) <= sequence_length:
            skipped += 1
            continue

        # timeseries_dataset_from_array with NO batching yet (batch_size=None)
        # data  [:-sequence_length] : input rows (shifted so last row has a target)
        # targets [sequence_length:]: the target that corresponds to the NEXT step
        ds = tf.keras.utils.timeseries_dataset_from_array(
            data            = X[:-sequence_length],
            targets         = y[sequence_length:],
            sequence_length = sequence_length,
            batch_size      = None,   # defer batching — we concatenate first
            shuffle         = False,  # never shuffle inside a station (breaks temporal order)
        )
        station_datasets.append(ds)

    if skipped:
        print(f"  Skipped {skipped} station(s) with fewer than {sequence_length+1} rows.")

    if not station_datasets:
        raise ValueError("No stations had enough rows to build sequences.")

    # Concatenate all per-station datasets into one flat stream
    combined = station_datasets[0]
    for ds in station_datasets[1:]:
        combined = combined.concatenate(ds)

    # NOW shuffle across stations (training only) and batch
    if shuffle:
        # buffer_size controls how many samples are held in memory for shuffling;
        # a large value gives better randomness but uses more RAM.
        combined = combined.shuffle(buffer_size=50_000, seed=42, reshuffle_each_iteration=True)

    combined = combined.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
    return combined


# ── STEP 3 : BUILD DATASETS ───────────────────────────────────────────────────
print("Building training dataset …")
train_ds = build_station_dataset(train_df, SEQUENCE_LENGTH, BATCH_SIZE, shuffle=True)

print("Building validation dataset …")
val_ds   = build_station_dataset(test_df,  SEQUENCE_LENGTH, BATCH_SIZE, shuffle=False)

# ── STEP 4 : SANITY CHECK ─────────────────────────────────────────────────────
for X_batch, y_batch in train_ds.take(1):
    print(f"\nSanity check:")
    print(f"  X batch shape : {X_batch.shape}")   # (BATCH_SIZE, SEQUENCE_LENGTH, N_FEATURES)
    print(f"  y batch shape : {y_batch.shape}")   # (BATCH_SIZE,)
    print(f"  Expected X    : ({BATCH_SIZE}, {SEQUENCE_LENGTH}, {N_FEATURES})")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Flatten, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Dropout, LSTM


model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQUENCE_LENGTH, N_FEATURES),
         recurrent_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.2),
    LSTM(64, return_sequences=False, recurrent_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dense(1, activation='linear')
])

# 2. Compile Model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='mse', 
              metrics=['mae'])

# 3. Callbacks to prevent overfitting
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-5),
    ModelCheckpoint('best_model.keras', save_best_only=True)
]

# 4. Train the Model!
print("Starting ConvLSTM2D Training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

In [ ]:
# Collect all true targets from the val dataset (already correctly aligned)
y_true_scaled = np.concatenate([y for _, y in val_ds], axis=0)
y_pred_scaled = model.predict(val_ds)

y_true_real = pt.inverse_transform(y_true_scaled.reshape(-1, 1))
y_pred_real = pt.inverse_transform(y_pred_scaled)

mae  = mean_absolute_error(y_true_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
r2   = r2_score(y_true_real, y_pred_real)
print(f"MAE: {mae:.2f}  RMSE: {rmse:.2f}  R²: {r2:.4f}")

In [ ]:
# Save the entire model (architecture, weights, and optimizer state)
model.save('final_convlstm_model.keras')
print("Model successfully saved!")

In [24]:
# # Check for missing values
# missing_data = df.isnull().sum()
# print("Missing values per column:\n", missing_data[missing_data > 0])

# # Summary statistics of numerical columns

# # Check temporal coverage
# print(f"Data ranges from {df['timestamp'].min()} to {df['timestamp'].max()}")

In [25]:
# Step 3: Keep latitude/longitude as numeric, drop original categorical columns
# X = df.drop(columns=["sr_wm2"])
# y = df["sr_wm2"]

# # Your existing train/test split remains the same
# split_idx = int(len(df) * 0.8)
# X_train = X.iloc[:split_idx].copy()
# X_test = X.iloc[split_idx:].copy()
# y_train = y.iloc[:split_idx].copy()
# y_test = y.iloc[split_idx:].copy()

# print("Encoded features:")
# print(X_train.columns.tolist())
# print(f"\nX_train shape: {X_train.shape}")
# print(f"X_test shape: {X_test.shape}")

In [26]:
# from tensorflow import keras
# from tensorflow.keras import layers

# # Create a shallow neural network
# model = keras.Sequential([
#     layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
#     layers.Dense(32, activation='relu'),
#     layers.Dense(1)  # Output layer for regression
# ])

# # Compile the model
# model.compile(
#     optimizer='adam',
#     loss='mean_squared_error',
#     metrics=['mae']
# )

# # Train the model
# history = model.fit(
#     X_train, y_train,
#     epochs=50,
#     batch_size=32,
#     validation_split=0.2,
#     verbose=2
# )

# # Test the model
# y_pred_nn = model.predict(X_test)

# # Evaluation
# rmse = np.sqrt(mean_squared_error(y_test, y_pred_nn))
# mae = mean_absolute_error(y_test, y_pred_nn)
# r2 = r2_score(y_test, y_pred_nn)

# print("\n--- Neural Network Results ---")
# print(f"RMSE: {rmse:.2f} W/m²")
# print(f"MAE:  {mae:.2f} W/m²")
# print(f"R² Score: {r2:.4f}")

# # Plot training history
# plt.figure(figsize=(12, 4))
# plt.subplot(1, 2, 1)
# plt.plot(history.history['loss'], label='Train Loss')
# plt.plot(history.history['val_loss'], label='Val Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.title('Training History - Loss')

# plt.subplot(1, 2, 2)
# plt.plot(history.history['mae'], label='Train MAE')
# plt.plot(history.history['val_mae'], label='Val MAE')
# plt.xlabel('Epoch')
# plt.ylabel('MAE')
# plt.legend()
# plt.title('Training History - MAE')
# plt.tight_layout()
# plt.show()